# Reddit Submission Dataset: Exploratory Data Analysis and Preprocessing

This notebook cleans, filters, and structurally analyzes submission data collected from the **r/ukraine** community on Reddit, focusing on communication dynamics, war reporting, and fundraising efforts. The primary goal is to build a single, clean, analysis-ready dataset `df_clean` and to evaluate its core architecture: schema reduction, missingness patterns, submission volume, engagement metrics, media vs. text content distribution, deduplication, and topical community flairs.

**Structure:**
- **Part 1: Environment Setup & Data Ingestion** — Mounting Google Drive storage and loading the raw multi-year Reddit submission dump into Pandas.
- **Part 2: Feature Pruning & Sparsity Analysis** — Profiling column missingness and eliminating uninformative, deprecated, or 100% empty platform attributes.
- **Part 3: Research-Driven Feature Selection** — Subsetting 35 target variables covering content semantics, engagement metrics, provenance, and community reach.
- **Part 4: Duplicate Analysis & Content De-noising** — Filtering micro-length crisis spam and resolving API multi-submission artifacts by preserving peak-engagement entries.
- **Part 5: Data Completeness & Structural Inspection** — Auditing `NaN` and zero frequencies, examining media post behavior (`selftext`), and evaluating native crosspost semantics (`crosspost_parent`).
- **Part 6: Data Type Normalization** — Inspecting the resulting schema and converting raw Unix epoch timestamps into standardized UTC datetime objects.
- **Part 7: Exploratory Data Analysis & Baseline Metrics (Task 2)** — Computing dataset volume, in-memory footprint, temporal coverage, and distributions across content formats, top flairs, and external referral domains.
- **Part 8: Cleaned Corpus Export** — Persisting the finalized, curated dataset into a CSV file for archival and downstream research tasks.

## 1. Environment Setup & Data Ingestion

Mounting Google Drive storage and reading the primary raw Reddit submission corpus.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import pandas as pd

ROOT = '/content/drive/MyDrive/H2_CSS/raw-data'
csv_path = os.path.join(ROOT, 'reddit_ukraine.csv')

df = pd.read_csv(csv_path)
df_reddit = df

print(df_reddit.shape)
df.head()

/tmp/ipykernel_531/4043307614.py:7: DtypeWarning: Columns (0,1,5,11,12,15,17,18,23,24,26,27,28,29,30,31,44,46,50,52,54,63,64,83,90,91,97,100,101,103,107,109,116,118,119,120,122,123,125,126) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


(349430, 127)


,allow_live_comments,archived,author,author_created_utc,author_flair_background_color,author_flair_css_class,author_flair_template_id,author_flair_text,author_flair_text_color,awarders,...,num_reports,removal_reason,report_reasons,saved,ups,user_reports,visited,updated_on,previous_selftext,all_awardings
0,False,False,[deleted],NaN,NaN,NaN,NaN,NaN,dark,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,True,False,Regrup,1.371221e+09,#d3d6da,NaN,a7506210-05ce-11e8-85e3-0e23378c264c,Kharkiv,dark,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,True,False,[deleted],NaN,NaN,NaN,NaN,NaN,dark,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,True,False,ceobrunswick,1.611180e+09,NaN,NaN,NaN,NaN,NaN,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,False,False,FenixWater75,1.624214e+09,NaN,NaN,NaN,NaN,NaN,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
raw_df = df_reddit

raw_rows, raw_cols = raw_df.shape

buffer = io.BytesIO()
raw_df.to_csv(buffer, index=False, encoding='utf-8')
raw_size_mb = buffer.tell() / (1024 ** 2)

raw_overview_data = [
    {"Metric": "Raw Dataset Shape", "Value": f"{raw_rows:,} rows × {raw_cols} columns"},
    {"Metric": "Total Raw Records (Rows)", "Value": f"{raw_rows:,}"},
    {"Metric": "Total Raw Features (Columns)", "Value": f"{raw_cols:,}"},
    {"Metric": "Raw Data Size (CSV Payload)", "Value": f"{raw_size_mb:.2f} MB"}
]

raw_overview_df = pd.DataFrame(raw_overview_data)

display(raw_overview_df.style.hide(axis='index').set_properties(**{'text-align': 'left'}))

Metric,Value
Raw Dataset Shape,"349,430 rows × 127 columns"
Total Raw Records (Rows),"349,430"
Total Raw Features (Columns),127
Raw Data Size (CSV Payload),764.78 MB


## 2. Feature Sparsity & High-Dimension Pruning

The raw ingestion yields an expansive schema of 127 metadata attributes. Many Reddit API fields represent internal moderation logs, obsolete feature flags, or endpoints that Reddit no longer populates. Here we evaluate feature-level non-null ratios to identify completely empty (0% filled) and deprecated platform variables.

In [ ]:
import pandas as pd

fill_pct = (df_reddit.notna().sum() / len(df_reddit) * 100).sort_values()
empty_count = df_reddit.notna().sum()

fill_summary = pd.DataFrame({
    'non_null_count': empty_count,
    'non_null_pct': fill_pct,
    'dtype': df_reddit.dtypes.astype(str)
}).sort_values('non_null_pct')

fully_empty = fill_summary[fill_summary['non_null_pct'] == 0]
print(f"Fully empty columns count: {len(fully_empty)}")
print(fully_empty)

top20_empty = fill_summary.head(20)
print("\nTop-20 least populated columns:")
print(top20_empty)

Fully empty columns count: 14
                    non_null_count  non_null_pct    dtype
approved_by                      0           0.0  float64
approved_at_utc                  0           0.0  float64
call_to_action                   0           0.0  float64
content_categories               0           0.0  float64
category                         0           0.0  float64
banned_at_utc                    0           0.0  float64
banned_by                        0           0.0  float64
likes                            0           0.0  float64
removed_by                       0           0.0  float64
mod_reason_title                 0           0.0  float64
view_count                       0           0.0  float64
top_awarded_type                 0           0.0  float64
mod_reason_by                    0           0.0  float64
mod_note                         0           0.0  float64

Top-20 least populated columns:
                    non_null_count  non_null_pct    dtype
approved_

### Justification for Dropping Initial Sparsity Subset (20 Columns)

* **100% Missing Technical / Deprecated Fields:** `approved_by`, `approved_at_utc`, `call_to_action`, `content_categories`, `category`, `banned_at_utc`, `banned_by`, `likes`, `removed_by`, `mod_reason_title`, `view_count`, `top_awarded_type`, `mod_reason_by`, `mod_note` contain exactly 0 non-null values across all 349k records.
* **Ultra-Sparse Feature Artifacts (< 0.03% Fill Rate):** `discussion_type` (1 row), `tournament_data` (1 row), `event_end` / `event_start` / `event_is_live` (4 rows each), and `removal_reason` (77 rows). These fields carry negligible statistical support for computational modeling.

In [ ]:
cols_to_drop = [
    'approved_by', 'approved_at_utc', 'call_to_action', 'content_categories',
    'category', 'banned_at_utc', 'banned_by', 'likes', 'removed_by',
    'mod_reason_title', 'view_count', 'top_awarded_type', 'mod_reason_by',
    'mod_note', 'discussion_type', 'tournament_data', 'event_end',
    'event_start', 'event_is_live', 'removal_reason'
]

initial_cols = df.shape[1]
df = df.drop(columns=cols_to_drop)

print(f"Initial columns count: {initial_cols}")
print(f"Columns dropped: {len(cols_to_drop)}")
print(f"Remaining columns: {df.shape[1]}")

Initial columns count: 127
Columns dropped: 20
Remaining columns: 107


### Inspecting Retained Candidate Features

Printing the remaining 107 features to map their semantic relevance to our computational study of community dynamics and donation calls.

In [ ]:
for i, col in enumerate(df.columns):
    print(f"{i + 1}. {col}")

1. allow_live_comments
2. archived
3. author
4. author_created_utc
5. author_flair_background_color
6. author_flair_css_class
7. author_flair_template_id
8. author_flair_text
9. author_flair_text_color
10. awarders
11. can_gild
12. can_mod_post
13. contest_mode
14. created_utc
15. distinguished
16. domain
17. edited
18. gilded
19. gildings
20. hidden
21. hide_score
22. id
23. is_created_from_ads_ui
24. is_crosspostable
25. is_meta
26. is_original_content
27. is_reddit_media_domain
28. is_robot_indexable
29. is_self
30. is_video
31. link_flair_background_color
32. link_flair_css_class
33. link_flair_richtext
34. link_flair_template_id
35. link_flair_text
36. link_flair_text_color
37. link_flair_type
38. locked
39. media
40. media_embed
41. media_only
42. name
43. no_follow
44. num_comments
45. num_crossposts
46. over_18
47. parent_whitelist_status
48. permalink
49. pinned
50. pwls
51. quarantine
52. removed_by_category
53. retrieved_on
54. retrieved_utc
55. score
56. secure_media
57. se

## Part 3: Analytical Dimension Grouping & Feature Subsetting

Rather than retaining internal ad-tracking flags and redundant rendering properties, we isolate **35 target variables** across six core analytical dimensions to support downstream modeling, network analysis, and rich content inspection:

1. **Content Semantics & Chronological Provenance:**
   * `id`: Unique submission identifier for primary key constraints and thread reconstruction.
   * `created_utc`: Publication epoch timestamp for temporal sequence and time-series modeling.
   * `title`, `selftext`: Core textual corpus used for NLP extraction, sentiment tracking, and fundraising keyword detection.
   * `permalink`: Permanent relative URL path enabling direct validation and qualitative thread inspection.

2. **Author Identity, Legitimacy & Social Capital:**
   * `author`: Account handle used to map prominent volunteer clusters and content creators.
   * `author_fullname`: Immutable Reddit account ID (`t2_*`) preserving user tracking across handle updates.
   * `author_flair_text`: Community verification tag (e.g., *Verified Volunteer*, regional labels) capturing user credibility.
   * `author_premium`: Boolean flag reflecting Reddit Premium status, often correlating with dedicated organizational profiles.

3. **Information Provenance, Media Modality & Format Categorization:**
   * `url`, `domain`: External destinations capturing payment endpoints (Monobank jars, PayPal, Patreon) and primary news outlets.
   * `link_flair_text`: Topic-level categorization assigned by submitters and community moderators (e.g., *Charity*, *WAR*, *News*).
   * `post_hint`: High-level submission classification provided by Reddit (`image`, `link`, `hosted:video`).
   * `is_self`: Strict boolean indicator separating textual discussions from external link or media submissions.
   * `is_video`, `is_gallery`: Media modality indicators identifying audiovisual materials and multi-image photo sets.
   * `thumbnail`: Preview thumbnail reference illustrating media richness and visual rendering.

4. **Community Resonance & Engagement Feedback Loops:**
   * `score`, `upvote_ratio`: Net community score and positive approval percentage.
   * `num_comments`: Conversational depth and public discussion activity.
   * `total_awards_received`, `gilded`: Aggregated gift counts and premium gold honors reflecting strong monetary or emotional resonance.

5. **Diffusion Dynamics, Moderation Flags & Reach:**
   * `num_crossposts`, `crosspost_parent`: Repost frequency and original submission provenance across Reddit networks.
   * `subreddit`, `subreddit_subscribers`: Subreddit identity and community subscriber scale at publication time.
   * `over_18`: Sensitive content tag (NSFW) separating combat-heavy or traumatic imagery from general civil updates.
   * `stickied`: Moderator pin flag indicating submissions boosted by official promotion.
   * `locked`: Administrative status showing whether discussion threads were disabled due to policy breaches.
   * `edited`: Timestamp or flag recording subsequent revisions made to donation links, details, or text bodies.
   * `is_original_content`: Submitter tag identifying firsthand, original reporting and visual materials.
   * `removed_by_category`: Moderation reason assigned when posts were suppressed by community moderators, AutoMod, or Reddit filters.

6. **Rich Visual Metadata & Micro-Donation Architectures:**
   * `preview`: Detailed JSON structures containing multi-resolution image resolutions, responsive crops, and source image URLs.
   * `gallery_data`: Complete itemized JSON rosters for multi-image photo dumps, receipts, and field reports.
   * `all_awardings`: Granular JSON records covering distinct badges, micro-transactions, and community-funded rewards attached to each submission.

In [ ]:
target_columns = [
    'id', 'created_utc', 'title', 'selftext', 'permalink',
    'author', 'author_fullname', 'author_flair_text', 'author_premium',
    'url', 'domain', 'link_flair_text', 'post_hint', 'is_self', 'is_video', 'is_gallery', 'thumbnail',
    'score', 'upvote_ratio', 'num_comments', 'total_awards_received', 'gilded',
    'num_crossposts', 'crosspost_parent', 'subreddit', 'subreddit_subscribers',
    'over_18', 'stickied', 'locked', 'edited', 'is_original_content', 'removed_by_category',
    'preview', 'gallery_data', 'all_awardings'
]
final_cols = [col for col in target_columns if col in df.columns]
df_clean = df[final_cols].copy()

print(f"Selected columns count: {df_clean.shape[1]}")
print("\nSelected column list:")
print(list(df_clean.columns))

Selected columns count: 35

Selected column list:
['id', 'created_utc', 'title', 'selftext', 'permalink', 'author', 'author_fullname', 'author_flair_text', 'author_premium', 'url', 'domain', 'link_flair_text', 'post_hint', 'is_self', 'is_video', 'is_gallery', 'thumbnail', 'score', 'upvote_ratio', 'num_comments', 'total_awards_received', 'gilded', 'num_crossposts', 'crosspost_parent', 'subreddit', 'subreddit_subscribers', 'over_18', 'stickied', 'locked', 'edited', 'is_original_content', 'removed_by_category', 'preview', 'gallery_data', 'all_awardings']


## 4. Deduplication & Noise Filtering

### 4.1. Exact Match Audits
Evaluating strict row duplicates and submission `id` integrity.

In [ ]:
full_duplicates = df_clean.duplicated().sum()
print(f"Full row duplicates: {full_duplicates}")

Full row duplicates: 0


In [ ]:
id_duplicates = df_clean.duplicated(subset=['id']).sum()
print(f"Duplicates by post ID: {id_duplicates}")

Duplicates by post ID: 0


### 4.2. Semantic Title Duplication Inspection

While primary IDs are unique, title-level duplicates total 23,072 entries. We sort and sample them to uncover the underlying generation mechanism (e.g., bots, API multi-posting, or recurring media headlines).

In [ ]:
title_dupes = df_clean.duplicated(subset=['title']).sum()
print(f"Duplicates by title: {title_dupes}")

Duplicates by title: 23072


In [ ]:
title_dupes_df = df_clean[df_clean.duplicated(subset=['title'], keep=False)]
title_dupes_df = title_dupes_df.sort_values(by=['title', 'created_utc'])
display(title_dupes_df[['title', 'author', 'subreddit', 'created_utc', 'score']].head(15))

,title,author,subreddit,created_utc,score
24780,!,f0rgotten,ukraine,1.645975e+09,6
45829,!,[deleted],ukraine,1.646330e+09,0
68665,!,greetingsfromfinland,ukraine,1.647174e+09,7195
290825,! URGENT REQUEST! From The Guys Fighting In Th...,ongand2,ukraine,1.715889e+09,2
290826,! URGENT REQUEST! From The Guys Fighting In Th...,ongand2,ukraine,1.715889e+09,2
290827,! URGENT REQUEST! From The Guys Fighting In Th...,ongand2,ukraine,1.715889e+09,2
290828,! URGENT REQUEST! From The Guys Fighting In Th...,ongand2,ukraine,1.715889e+09,367
8937,!!!!,ipunchvagina,ukraine,1.645761e+09,0
29975,!!!!,Sorry-Sherbet-2422,ukraine,1.646046e+09,1
23700,"""2nd strongest military in the world""",ToxicAbility,ukraine,1.645963e+09,43


### 4.3. Data Cleaning Policy Formulation

Inspection of title duplicates reveals two distinct phenomena:
1. **Crisis-Onset Panic Spam:** Short uninformative exclamations (`!`, `!!!!`) posted across late February 2022 with zero informational or analytical value.
   * *Remedy:* Filter out entries where `len(title) <= 5`.
2. **API Double-Submission / Client Retries:** Identical authors publishing identical titles to the same subreddit within milliseconds (e.g., repeated fundraising appeals by `ongand2`).
   * *Remedy:* Deduplicate by `['title', 'author', 'subreddit']`, sorting by `score` descending so that the submission capturing actual audience engagement is preserved.

In [ ]:
df_clean = df_clean[df_clean['title'].astype(str).str.len() > 5]
df_clean = df_clean.sort_values(by=['title', 'author', 'score'], ascending=[True, True, False])
df_clean = df_clean.drop_duplicates(subset=['title', 'author', 'subreddit'], keep='first')

print(f"Dataset shape after cleaning text noise and duplicates: {df_clean.shape}")

Dataset shape after cleaning text noise and duplicates: (333290, 35)


## 5. Missingness Architecture & Structural Audits

### 5.1. Quantifying Missingness and Zero Value Proportions
We compute null ratios and numerical zero frequencies to diagnose data sparsity patterns.

In [ ]:
nan_percentages = (df_clean.isnull().sum() / len(df_clean)) * 100

numeric_cols = df_clean.select_dtypes(include=['number']).columns
zero_percentages = (df_clean[numeric_cols] == 0).sum() / len(df_clean) * 100

stats_df = pd.DataFrame({
    'NaN (%)': nan_percentages,
    'Zeros (%)': zero_percentages
}).fillna(0)

display(stats_df[(stats_df['NaN (%)'] > 0) | (stats_df['Zeros (%)'] > 0)].sort_values(by='NaN (%)', ascending=False))

,NaN (%),Zeros (%)
crosspost_parent,99.479432,0.000000
gallery_data,97.367158,0.000000
is_gallery,95.603528,0.000000
author_flair_text,84.442978,0.000000
all_awardings,77.301449,0.000000
selftext,62.544031,0.000000
post_hint,57.115425,0.000000
preview,56.991209,0.000000
removed_by_category,48.328483,0.000000
author_fullname,14.740016,0.000000


A closer inspection of duplicate titles highlights two primary patterns alongside deleted accounts: crisis-onset emotional spam and technical API artifacts. Short, panic-driven headlines (e.g., '!', '!!!!') posted during the initial invasion outbreak in early 2022 carry negligible informational value and near-zero engagement, justifying a threshold filter for titles shorter than five characters. Meanwhile, near-identical submissions published in rapid succession by the same author—often caused by network lags or Reddit API multi-submission retries—are deduplicated across title, author, and subreddit, preserving the single entry with the highest score to accurately capture peak community engagement. Finally, submissions attributed to [deleted] authors are intentionally retained; while account handles are purged upon profile deactivation, the posts themselves preserve valid discussion metadata, titles, and engagement scores essential for our aggregate analysis.

### 5.2. Qualitative Inspection of `selftext` Missingness

The `selftext` field exhibits ~62.5% missingness. On Reddit, submissions are bifurcated into **Self-posts** (text discussions) and **Link/Media posts** (photos, combat footage, news URLs). The lack of `selftext` is natural for media submissions where context is encapsulated entirely in `title` and `url`.

In [ ]:
valid_texts = df_clean[
    (df_clean['selftext'] != "") &
    (~df_clean['selftext'].isin(['[removed]', '[deleted]']))
]

samples = valid_texts.sample(n=5, random_state=42)

for i, (idx, row) in enumerate(samples.iterrows(), 1):
    print(f"--- SAMPLE {i} ---")
    print(f"TITLE: {row['title']}")
    text_preview = str(row['selftext'])[:400].replace('\n', ' ')
    print(f"TEXT: {text_preview}...")
    print("="*70 + "\n")

--- SAMPLE 1 ---
TITLE: Desperation Unveiled: Russia's Bold Threat to Seize $13 Billion Oligarch's Assets : City Telegraph
TEXT: nan...

--- SAMPLE 2 ---
TITLE: Our Hero Ukraine Pilot
TEXT: nan...

--- SAMPLE 3 ---
TITLE: Doing my part. Slava Ukraini
TEXT: nan...

--- SAMPLE 4 ---
TITLE: That's how units of the Armed Forces of the Armed Forces of Ukraine landed the Russian reconnaissance drone "Orlan-10"
TEXT: nan...

--- SAMPLE 5 ---
TITLE: Crimea's Intensified Resistance Movement
TEXT: nan...



Submissions with missing selftext values are intentionally retained, as they typically represent media- or link-driven posts (such as combat footage, infographics, or photo reports) rather than text discussions. In these instances, the entire context—often including the fundraising purpose or video reports of military aid deliveries—is encapsulated directly within the submission title and external URL.

### 5.3. Qualitative Inspection of `crosspost_parent`

The attribute `crosspost_parent` is sparse (99.48% `NaN`). Examining populated instances reveals Reddit Fullnames (`t3_<id>`), identifying explicit reposts from other communities.

In [ ]:
crosspost_examples = df_clean[df_clean['crosspost_parent'].notna()][[
    'id', 'title', 'author', 'crosspost_parent', 'score'
]].head(5)
display(crosspost_examples)

,id,title,author,crosspost_parent,score
14521,t1ln82,"""A house in Teremkivska transmitting light sig...",Visible_Coyote_4000,t3_t1lj3x,69
2164,svmwo5,"""Explosion"" occurs near the DPR HQ building an...",MuzzleO,t3_svlrqv,15
2721,sx3ab7,"""Eyewitnesses report that in Luhansk the milit...",MuzzleO,t3_sx1a15,16
12633,t1cs9m,"""I condemn in the strongest possible way russi...",woosal1337,t3_t1cr0d,62
23168,t2ktwt,"""I hope they are fine""",digging_for_1_Gon4_2,t3_t2eevh,591


In `r/ukraine`, moderation guidelines strictly encourage direct primary-source submissions over native Reddit crossposts. Hence, `NaN` correctly indicates an original, standalone submission rather than missing information.

## 6. Schema Profiling & Data Type Normalization

### 6.1. Current Schema State
Reviewing column datatypes and non-null counts prior to normalization.

In [ ]:
print(df_clean.info())

dtype_summary = pd.DataFrame({
    'Column': df_clean.columns,
    'Dtype': df_clean.dtypes.values,
    'Sample Value': [df_clean[col].dropna().iloc[0] if not df_clean[col].dropna().empty else None for col in df_clean.columns]
})

display(dtype_summary)

<class 'pandas.core.frame.DataFrame'>
Index: 333290 entries, 244404 to 348210
Data columns (total 35 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   id                     333290 non-null  object 
 1   created_utc            333290 non-null  float64
 2   title                  333290 non-null  object 
 3   selftext               124837 non-null  object 
 4   permalink              333290 non-null  object 
 5   author                 333290 non-null  object 
 6   author_fullname        284163 non-null  object 
 7   author_flair_text      51850 non-null   object 
 8   author_premium         284163 non-null  object 
 9   url                    316554 non-null  object 
 10  domain                 316533 non-null  object 
 11  link_flair_text        311824 non-null  object 
 12  post_hint              142930 non-null  object 
 13  is_self                333290 non-null  bool   
 14  is_video               333290 non-nu

,Column,Dtype,Sample Value
0,id,object,13iqk65
1,created_utc,float64,1684199167.0
2,title,object,\nBattle for Cyclops. SERIES 5. The enemy re...
3,selftext,object,[removed]
4,permalink,object,/r/ukraine/comments/13iqk65/battle_for_cyclops...
5,author,object,Res-Horizon
6,author_fullname,object,t2_lyiq4pxr
7,author_flair_text,object,:FlagUA: Ukraine Media
8,author_premium,object,True
9,url,object,https://v.redd.it/7p4186t5f30b1


### 6.2. Temporal Normalization

Converting the numeric Unix epoch timestamp (`created_utc`) from raw `float64` seconds into native `datetime64[ns]` format to enable chronological filtering and temporal aggregation.

In [ ]:
df_clean['created_utc'] = pd.to_datetime(df_clean['created_utc'], unit='s')

print(f"New data type: {df_clean['created_utc'].dtype}")
print("\nFirst 3 timestamps for verification:")
print(df_clean['created_utc'].head(3))

New data type: datetime64[ns]

First 3 timestamps for verification:
244404   2023-05-16 01:06:07
293278   2024-06-09 09:09:36
309925   2024-11-20 07:04:31
Name: created_utc, dtype: datetime64[ns]


## 7. Baseline Metrics & Corpus Exploration

### 7.1. Global Metrics Summary
Evaluating total record volume, memory usage, earliest and latest publication timestamps, and total span in months (verifying the 6+ months assignment requirement).

In [ ]:
total_rows = len(df_clean)
memory_mb = df_clean.memory_usage(deep=True).sum() / (1024 ** 2)
min_date = df_clean['created_utc'].min()
max_date = df_clean['created_utc'].max()
timespan_months = (max_date - min_date).days / 30.44

metrics_summary = pd.DataFrame({
    'Metric': [
        'Total Records (Rows)', 'Total Columns', 'In-Memory Size (MB)',
        'Earliest Post (UTC)', 'Latest Post (UTC)', 'Time Range (Months)'
    ],
    'Value': [
        f"{total_rows:,}", str(df_clean.shape[1]), f"{memory_mb:.2f} MB",
        str(min_date), str(max_date), f"{timespan_months:.1f} months"
    ]
})

display(metrics_summary)

,Metric,Value
0,Total Records (Rows),"333,290"
1,Total Columns,35
2,In-Memory Size (MB),686.21 MB
3,Earliest Post (UTC),2022-02-01 00:06:00
4,Latest Post (UTC),2026-03-02 22:48:19
5,Time Range (Months),48.9 months


### 7.2. Distribution by Content Format (Text vs. Media/Links)

Classifying posts into text submissions vs. external links/media reveals that over 90% of `r/ukraine` communications are media- or link-driven, reflecting a heavy reliance on visual war documentation and external news dissemination.

In [ ]:
df_clean['post_type'] = df_clean['selftext'].apply(
    lambda x: 'Text Submission' if pd.notna(x) and x not in ['[removed]', '[deleted]', ''] else 'Media / External Link'
)
print(" Distribution by Content Type ")
print(df_clean['post_type'].value_counts(normalize=False))
print("\nPercentages:")
print((df_clean['post_type'].value_counts(normalize=True) * 100).round(2))

 Distribution by Content Type 
post_type
Media / External Link    301851
Text Submission           31439
Name: count, dtype: int64

Percentages:
post_type
Media / External Link    90.57
Text Submission           9.43
Name: proportion, dtype: float64


### 7.3. Categorical Distribution by Community Flairs

Auditing top post tags (`link_flair_text`) to assess topical distribution across war reporting, community Q&A, and verified updates.

In [ ]:
print("\nTop 10 Categories")
flair_counts = df_clean['link_flair_text'].value_counts(dropna=False).head(10).reset_index()
flair_counts.columns = ['Flair', 'Post Count']
flair_counts['Share (%)'] = (flair_counts['Post Count'] / total_rows * 100).round(2)
display(flair_counts)


Top 10 Categories


,Flair,Post Count,Share (%)
0,News,67112,20.14
1,WAR,44316,13.30
2,Trustworthy News,33792,10.14
3,Discussion,31230,9.37
4,Social Media,27479,8.24
5,Question,22678,6.80
6,NaN,21466,6.44
7,Media,18625,5.59
8,Russian-Ukrainian War,12035,3.61
9,WAR CRIME,9949,2.99


The NaN row appears in the top categories simply because adding a flair to a post on Reddit is optional, so some users publish without selecting a topic. We used dropna=False in the code to count these untagged posts as a separate group rather than hiding them. This shows that only 6.44% of the posts lack a flair, meaning over 93% of the dataset is properly tagged and ready for analysis.

### 7.4. Referral Source Landscape (Top Domains)

Examining external web sources to profile community reliance on native Reddit hosting (`i.redd.it`, `v.redd.it`), social feeds (Twitter/X, YouTube), and dedicated Ukrainian defense and media outlets (`mil.in.ua`, `kyivindependent.com`).

In [ ]:
print("\nTop 10 Domains")
domain_counts = df_clean['domain'].value_counts(dropna=False).head(10).reset_index()
domain_counts.columns = ['Domain', 'Count']
display(domain_counts)


Top 10 Domains


,Domain,Count
0,self.ukraine,77391
1,i.redd.it,44912
2,v.redd.it,30530
3,twitter.com,17104
4,NaN,16757
5,reddit.com,15149
6,youtu.be,12526
7,youtube.com,12089
8,mil.in.ua,8476
9,kyivindependent.com,5800


The NaN row appears in the top domains table because Reddit posts are divided into link submissions and self-contained text posts. When a user writes a direct text post without attaching an external website or link, Reddit often leaves the domain field empty rather than assigning it to self.ukraine. By including dropna=False, the calculation honestly reflects these 16,757 entries (around 5% of the total dataset) as posts that do not reference outside websites.

### 7.5. Dataset Dimensions & Storage Overview


In [ ]:
num_rows, num_cols = df_clean.shape

buffer = io.BytesIO()
df_clean.to_csv(buffer, index=False, encoding='utf-8')
csv_size_mb = buffer.tell() / (1024 ** 2)

overview_data = [
    {"Metric": "Dataset Shape", "Value": f"{num_rows:,} rows × {num_cols} columns"},
    {"Metric": "Total Records (Rows)", "Value": f"{num_rows:,}"},
    {"Metric": "Total Features (Columns)", "Value": f"{num_cols:,}"},
    {"Metric": "CSV Export File Size", "Value": f"{csv_size_mb:.2f} MB"}
]

dim_overview_df = pd.DataFrame(overview_data)

display(dim_overview_df.style.hide(axis='index').set_properties(**{'text-align': 'left'}))

Metric,Value
Dataset Shape,"333,290 rows × 36 columns"
Total Records (Rows),"333,290"
Total Features (Columns),36
CSV Export File Size,355.50 MB


#### 7.6 Frequency Overview of Data Types

We summarize the distribution of data types across the schema of `df_clean`, presenting the absolute column count, relative frequency, and the corresponding column names for each `dtype`.

In [ ]:
dtype_mapping = {}
for col, dtype in df_clean.dtypes.items():
    dtype_str = str(dtype)
    dtype_mapping.setdefault(dtype_str, []).append(col)

total_cols = len(df_clean.columns)
dtype_summary = []

for dtype_str, cols in dtype_mapping.items():
    count = len(cols)
    pct = (count / total_cols) * 100
    dtype_summary.append({
        'Data Type': dtype_str,
        'Column Count': count,
        'Frequency (%)': f"{pct:.1f}%",
        'Columns': ", ".join(cols),
        '_sort_key': count
    })

dtype_df = pd.DataFrame(dtype_summary).sort_values(by='_sort_key', ascending=False).drop(columns=['_sort_key'])

display(dtype_df.style.hide(axis='index').set_properties(**{'text-align': 'left'}))

Data Type,Column Count,Frequency (%),Columns
object,23,63.9%,"id, title, selftext, permalink, author, author_fullname, author_flair_text, author_premium, url, domain, link_flair_text, post_hint, is_gallery, thumbnail, crosspost_parent, subreddit, edited, is_original_content, removed_by_category, preview, gallery_data, all_awardings, post_type"
bool,5,13.9%,"is_self, is_video, over_18, stickied, locked"
float64,4,11.1%,"upvote_ratio, total_awards_received, gilded, num_crossposts"
int64,3,8.3%,"score, num_comments, subreddit_subscribers"
datetime64[ns],1,2.8%,created_utc


## 8. Export Cleaned Dataset

Persisting the cleaned and structured research corpus into CSV format for project archive, git synchronization, and subsequent quantitative modeling.

In [ ]:
output_path = os.path.join(ROOT, 'reddit_ukraine_cleaned.csv')
df_clean.to_csv(output_path, index=False)

print(f"Cleaned dataset saved successfully to: {output_path}")
print(f"Disk file size: {os.path.getsize(output_path) / (1024 ** 2):.2f} MB")

Cleaned dataset saved successfully to: /content/drive/MyDrive/H2_CSS/raw-data/reddit_ukraine_cleaned.csv
Disk file size: 355.50 MB
